# 7 бинарных классов для определения болезней листьев томатов

Для transfer learning используется модель ResNet50

## Подключение

In [1]:
!pip install wandb -qU
!pip install torchmetrics

import os
import shutil
import random

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import wandb

import torchvision
from torchvision import datasets, models, transforms

import albumentations as A

from sklearn.metrics import precision_recall_curve, auc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 6.8 MB/s eta 0:00:00


In [2]:
!wget -nc -O dataset.zip https://www.dropbox.com/scl/fo/3plo5qmx1o2sq7rrvds2c/h?rlkey=eupcg0up7ezxasdaabs4ywole&dl=1

--2023-12-20 17:18:57--  https://www.dropbox.com/scl/fo/3plo5qmx1o2sq7rrvds2c/h?rlkey=eupcg0up7ezxasdaabs4ywole
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc66d71324c7079a057a79e78ec2.dl.dropboxusercontent.com/zip_download_get/BtKF6e2cLbLeWywIbPN42gZuYd0uyXwkD7U9PeKlS4udD47FthOUxSwTKMcZZECbiB06sowbK5KYjE0gHE6IRmcCCK9HOPbtol1cv_KDNkOSgw# [following]
--2023-12-20 17:19:05--  https://uc66d71324c7079a057a79e78ec2.dl.dropboxusercontent.com/zip_download_get/BtKF6e2cLbLeWywIbPN42gZuYd0uyXwkD7U9PeKlS4udD47FthOUxSwTKMcZZECbiB06sowbK5KYjE0gHE6IRmcCCK9HOPbtol1cv_KDNkOSgw
Resolving uc66d71324c7079a057a79e78ec2.dl.dropboxusercontent.com (uc66d71324c7079a057a79e78ec2.dl.dropboxusercontent.com)... 162.125.1.15, 2620:100:6016:15::a27d:10f
Connecting to uc66d71324c7079a057a79e78ec2.dl.dropboxusercontent.com (uc66

In [3]:
!unzip -n -d dataset dataset.zip

Выходные данные были обрезаны до нескольких последних строк (5000).
 extracting: dataset/train/Tomato_Septoria_leaf_spot/56e5932f-b715-40e3-b9b8-caef91f725db___JR_Sept.L.S 2568.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/7c464f54-8a3e-45ad-a900-d7164e44cc0d___JR_Sept.L.S 8534.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/56d9c252-fe48-4a39-8264-8bfce1f8ba2a___JR_Sept.L.S 2562.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/53f04b6f-afa0-44ca-906b-477462819a65___JR_Sept.L.S 8546.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/7a74368f-379b-4bb8-a8cb-a21b86ef8d63___JR_Sept.L.S 2590.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/8a5a6fa0-17f1-42da-b7ee-406783e8666b___JR_Sept.L.S 8411.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/6bb03663-2989-4c0b-8aba-bedc9e3430eb___JR_Sept.L.S 2607.JPG  
 extracting: dataset/train/Tomato_Septoria_leaf_spot/8cbaba7f-8b0b-4eea-a4ed-afd375a7c55a___JR_Sept.L.S 2720.JPG  
 extracting:

In [4]:
for i in os.listdir('/content/dataset/train/tomato_leaf'):
    for j in os.listdir('/content/dataset/train/tomato_leaf' + '/' +i):
        for k in os.listdir('/content/dataset/train/tomato_leaf' + '/' +i + '/' + j):
            os.remove('/content/dataset/train/tomato_leaf' + '/' +i + '/' + j + '/' + k)
        os.rmdir('/content/dataset/train/tomato_leaf' + '/' +i + '/' + j)
    os.rmdir('/content/dataset/train/tomato_leaf' + '/' +i )
os.rmdir('/content/dataset/train/tomato_leaf')

## Функция смешивания

In [5]:
def mix_datasets(comparison_folder):
    directory = ['train', 'test']
    for dir in directory:
        path = os.path.join('/content', dir)
        os.mkdir(path)
        path_another = path + '/another'
        os.mkdir(path_another)
        path_comparison = path + '/comparison_folder'
        os.mkdir(path_comparison)
        move_list = []
        for root_dir in os.listdir('/content/dataset/' + dir):
            road = os.path.join('/content/dataset/' + dir, root_dir)
            if root_dir != comparison_folder:
                move_list += [(road + '/' + img, path_another + '/' + img) for img in os.listdir(road) if img.endswith(('.jpg', '.jpeg', '.JPG'))]
            else:
                for img in os.listdir(road):
                    shutil.move(road + '/' + img, path_comparison + '/' + img)
        random.shuffle(move_list)
        for move in move_list:
            shutil.move(move[0], move[1])

## Подготовка данных

In [7]:
resnet_transforms = models.ResNet50_Weights.IMAGENET1K_V1.transforms()

In [8]:
train_data = datasets.ImageFolder('/content/train', transform=resnet_transforms)
test_data = datasets.ImageFolder('/content/test', transform=resnet_transforms)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
class_to_idx = train_data.class_to_idx
class_to_idx

{'another': 0, 'comparison_folder': 1}

In [11]:
train_size = int(len(train_data) * 0.8)
# в валидационную 20%
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(train_data, [train_size, val_size])

In [12]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=256, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=256, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=256, shuffle=False)

## Бинарные классы для каждого вида болезни

### Tomato_Bacterial_spot vs all the rest

In [6]:
mix_datasets('Tomato_Bacterial_spot')

In [17]:
model = models.resnet50(pretrained=True)



# Замораживаем все слои модели
for param in model.parameters():
    param.requires_grad = False

# Заменяем последний слой на новый для нашей задачи классификации
num_classes = len(class_to_idx)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

# Определение функции потерь и оптимизатора
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

num_epochs = 20

best_threshold_train_list =[]


wandb.init(project='test_binary_class_tomatos', name='Tomato_Bacterial_spot',
        config={
        "epochs": num_epochs
    })


for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0

    all_probs = []
    all_labels = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()

        probs = torch.softmax(outputs, dim=1).cpu().detach().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().detach().numpy())

    train_loss /= len(train_data)
    train_acc = train_correct / len(train_data)
    all_probs = np.vstack(all_probs)
    train_precision, train_recall, train_thresholds = precision_recall_curve(all_labels, all_probs[:, 1])
    pr_auc_train = auc(train_recall, train_precision)
    best_threshold_train = train_thresholds[np.argmin(train_precision - train_recall)]
    best_threshold_train_list.append(best_threshold_train)

    wandb.log({"Train Recall-Precision Curve": wandb.plot.pr_curve(all_labels, all_probs)}, step=epoch)
    model.eval()
    val_loss = 0.0
    val_correct = 0

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()

            probs = torch.softmax(outputs, dim=1).cpu().detach().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().detach().numpy())

    val_loss /= len(val_data)
    val_acc = val_correct / len(val_data)
    all_probs = np.vstack(all_probs)
    val_precision, val_recall, val_thresholds = precision_recall_curve(all_labels, all_probs[:, 1])
    pr_auc_val = auc(val_recall, val_precision)

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Best Threshold (Train): {best_threshold_train}")
    wandb.log({'train_loss': train_loss, 'train_acc': train_acc, 'pr_auc': pr_auc_train}, step=epoch)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | PR AUC: {pr_auc_train:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | PR AUC: {pr_auc_val:.4f}")
    wandb.log({'val_loss': val_loss, 'val_acc': val_acc, 'pr_auc': pr_auc_val}, step=epoch)
    print('-' * 60)


    wandb.log({"Val Recall-Precision Curve": wandb.plot.pr_curve(all_labels, all_probs)}, step=epoch)

# Тестирование модели
model.eval()
test_correct = 0

all_probs = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        all_probs = torch.softmax(outputs, dim=1).cpu().detach().numpy()
        all_labels = labels.cpu().numpy()
        test_correct += (outputs.argmax(dim=1) == labels).sum().item()

test_acc = test_correct / len(test_data)
all_probs = np.vstack(all_probs)
test_precision, test_recall, test_thresholds = precision_recall_curve(all_labels, all_probs[:, 1])
pr_auc_test = auc(test_recall, test_precision)
#best_threshold_test = test_thresholds[np.argmax(test_precision - test_recall)]

wandb.log({'test_acc': test_acc, 'pr_auc_test': pr_auc_test})
print(f"Test Acc: {test_acc:.4f} | PR AUC: {pr_auc_test:.4f}")

wandb.log({"Test Recall-Precision Curve": wandb.plot.pr_curve(all_labels, all_probs)})

# логируем лучшие отсечки
wandb.log({'best_thresholds_train': best_threshold_train_list})

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/20
Best Threshold (Train): 0.006353848148137331
Train Loss: 0.4596 | Train Acc: 0.8520 | PR AUC: 0.1587
Val Loss: 0.3661 | Val Acc: 0.8388 | PR AUC: 0.5617
------------------------------------------------------------
Epoch 2/20
Best Threshold (Train): 0.009246124885976315
Train Loss: 0.2997 | Train Acc: 0.8712 | PR AUC: 0.5945
Val Loss: 0.2898 | Val Acc: 0.8398 | PR AUC: 0.8052
------------------------------------------------------------
Epoch 3/20
Best Threshold (Train): 0.0052164578810334206
Train Loss: 0.2238 | Train Acc: 0.9036 | PR AUC: 0.8262
Val Loss: 0.2293 | Val Acc: 0.8969 | PR AUC: 0.8683
------------------------------------------------------------
Epoch 4/20
Best Threshold (Train): 0.002740852301940322
Train Loss: 0.1897 | Train Acc: 0.9240 | PR AUC: 0.8716
Val Loss: 0.2061 | Val Acc: 0.9163 | PR AUC: 0.8856
------------------------------------------------------------
Epoch 5/20
Best Threshold (Train): 0.001135660451836884
Train Loss: 0.1686 | Train Acc: 0.9347 | PR